# 🚀 ClawSouls — Avatar API Server (FastAPI + Cloudflared)

Sobe um servidor **FastAPI** no Colab que gera avatares por requisição HTTP.

**Fluxo:**
1. Rode este notebook (com GPU)
2. Copie a URL pública do Cloudflared
3. Envie POST `/generate` com os atributos da soul
4. Receba a imagem gerada em base64

---

## Pré-requisitos

- `Runtime > Change runtime type > T4 GPU`
- Não precisa do repositório clonado (tudo é self-contained)


In [ ]:
import torch
print(f"✅ PyTorch {torch.__version__} carregado")
print(f"✅ GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    total_mem = getattr(props, 'total_mem', getattr(props, 'total_memory', 0))
    print(f"   VRAM: {total_mem / 1e9:.1f} GB")
else:
    print("⚠️  Nenhuma GPU detectada!")


In [ ]:
!pip install -q fastapi uvicorn cloudflared pydantic pillow
!pip install -q diffusers[torch] transformers accelerate torch torchvision safetensors

print("✅ Dependências instaladas!")


---

## Configuração do Servidor

In [ ]:
# ═══════════════════════════════════════════════════════
# TOGGLE DE MODELO
# ═══════════════════════════════════════════════════════

MODELO = "sdxl"  # ← "turbo" ou "sdxl"

SECRET_TOKEN = "clawsouls-secret"  # ← Troque para algo seguro!

TOTAL_STEPS = 15  # Override global de steps (use 4-6 para turbo, 20-30 para sdxl)
TOTAL_GUIDANCE = 7.5  # Override global de guidance

# Catálogo
CATALOGO = {
    "turbo": {"model_id": "Tongyi-MAI/Z-Image-Turbo", "default_steps": 4, "default_guidance": 1.0, "width": 512, "height": 768, "variant": "fp16"},
    "sdxl":   {"model_id": "stabilityai/stable-diffusion-xl-base-1.0", "default_steps": 25, "default_guidance": 7.5, "width": 512, "height": 768, "variant": "fp16"},
}

assert MODELO in CATALOGO, f"Modelo '{MODELO}' não encontrado. Use: {list(CATALOGO.keys())}"
cfg = CATALOGO[MODELO]
MODEL_ID = cfg["model_id"]

print(f"🔧 Modelo: {MODELO} ({MODEL_ID})")
print(f"   Steps: {TOTAL_STEPS}, Guidance: {TOTAL_GUIDANCE}")


---

## Código do Servidor (FastAPI)

In [ ]:
import io, base64, time, os, threading
from datetime import datetime
from typing import Optional

from pydantic import BaseModel
from fastapi import FastAPI, HTTPException, Header
from fastapi.responses import StreamingResponse

import torch
from diffusers import AutoPipelineForText2Image

# ═══════════════════════════════════════════════════════
# PROMPT ENGINE (mesma lógica do batch generator)
# ═══════════════════════════════════════════════════════

EMOJI_HINTS = {
    "🔬": "scientific goggles, lab coat details",
    "🕵️": "detective hat, trench coat",
    "🌟": "sparkles, star-shaped accessories",
    "⚡": "electric energy aura, lightning motifs",
    "🧘": "lotus position, meditation beads, serene",
    "🤖": "mechanical parts, circuit patterns",
    "🏴‍☠️": "pirate bandana, adventurous look",
    "💻": "techwear, holographic screen elements",
    "🎤": "microphone, stage lights, glamorous",
    "🌳": "nature elements, leaves, organic flowing design",
    "🕶️": "sunglasses, cool demeanor",
    "😈": "mischievous grin, horns, dark aesthetic",
    "👽": "alien features, cosmic glow",
    "🐉": "dragon scales, mythical aura",
    "🦊": "fox ears, cunning expression",
    "🐱": "cat ears, playful whiskers",
    "👁️": "mystical third eye, all-seeing aura",
    "💀": "skull motifs, dark mysticism",
    "🎭": "theater mask, dramatic duality",
}

DOMAIN_ACCENTS = {
    "tech": "circuit patterns, holographic UI elements",
    "philosophy": "ancient scrolls, ethereal glow",
    "science": "molecular structures, lab equipment details",
    "arts": "paint splashes, creative chaos",
    "history": "ancient runes, time-worn textures",
    "literature": "floating text, book pages",
    "pop-culture": "retro gaming elements, neon signs",
    "sports": "athletic build, competitive energy",
    "business": "sharp suit, corporate confidence",
    "psychology": "thoughtful gaze, abstract mind visuals",
}


def build_prompt(soul: dict) -> str:
    creature = soul.get("creature", "mysterious entity")
    vibe = soul.get("vibe", "enigmatic")
    emoji = soul.get("emoji", "")
    humor = soul.get("humor", 50)
    formality = soul.get("formality", 50)
    vibe_style = soul.get("vibeStyle", "concise")
    knowledge_domains = soul.get("knowledgeDomains", [])
    emotional_range = soul.get("emotionalRange", 50)
    agreeableness = soul.get("agreeableness", 50)
    extraversion = soul.get("extraversion", 50)
    openness = soul.get("openness", 70)
    neuroticism = soul.get("neuroticism", 30)

    is_high_formality = formality > 65
    is_playful = humor > 65
    is_minimal = vibe_style == "minimal"
    is_concise = vibe_style == "concise"
    is_dramatic = emotional_range > 75 or vibe_style == "dramatic"
    is_tech = any(d in ("tech", "science") for d in knowledge_domains)

    art_style = "cyberpunk digital illustration"
    if is_high_formality:
        art_style = "elegant digital painting, Renaissance lighting"
    elif is_playful:
        art_style = "colorful anime-inspired digital art, vibrant"
    elif is_minimal:
        art_style = "minimalist vector art, clean lines, geometric"
    elif is_tech:
        art_style = "sci-fi concept art, holographic elements"
    elif is_dramatic:
        art_style = "cinematic digital painting, dramatic chiaroscuro lighting"

    atmosphere = "dark atmospheric background with neon accents"
    if agreeableness > 70:
        atmosphere = "warm, inviting background with soft golden light"
    elif neuroticism > 60:
        atmosphere = "unstable, glitching background with fractured light"
    elif extraversion > 70:
        atmosphere = "dynamic, energetic background with bold colors"
    elif openness > 75:
        atmosphere = "dreamy, surreal background with cosmic elements"

    expression = "calm, confident expression"
    if neuroticism > 60:
        expression = "tense, alert expression"
    elif extraversion > 70:
        expression = "bright, engaging smile"
    elif agreeableness > 70:
        expression = "gentle, warm expression"
    elif openness > 70:
        expression = "curious, contemplative gaze"
    elif humor > 70:
        expression = "sly, playful smirk"

    descriptors = [creature, vibe]
    if expression != "calm, confident expression":
        descriptors.append(expression)
    descriptors.append(art_style)
    if not is_concise and not is_minimal:
        descriptors.append(f"vibe: {vibe_style}")
    if emoji and emoji in EMOJI_HINTS:
        descriptors.append(EMOJI_HINTS[emoji])
    for domain in knowledge_domains:
        if domain in DOMAIN_ACCENTS:
            descriptors.append(DOMAIN_ACCENTS[domain])
    descriptors.append("unique, one-of-a-kind character design")

    prompt = (
        "close-up portrait, centered, detailed face, " +
        "professional character art of " + creature + ", " +
        ", ".join(descriptors[1:]) + ", " +
        atmosphere + ", highly detailed, 4k, masterpiece"
    )
    return prompt.strip()


def build_negative_prompt() -> str:
    return (
        "blurry, low quality, deformed, ugly, duplicate, disfigured, "
        "bad anatomy, bad proportions, extra limbs, mutated hands, "
        "text, watermark, signature, logo, "
        "photorealistic, 3d render, "
        "nude, NSFW, gore"
    )


# ═══════════════════════════════════════════════════════
# FastAPI App
# ═══════════════════════════════════════════════════════

class GenerateRequest(BaseModel):
    name: str
    creature: Optional[str] = "mysterious entity"
    vibe: Optional[str] = "enigmatic"
    emoji: Optional[str] = ""
    humor: Optional[int] = 50
    formality: Optional[int] = 50
    emojiUsage: Optional[int] = 20
    knowledgeDomains: Optional[list] = []
    emotionalRange: Optional[int] = 50
    vibeStyle: Optional[str] = "concise"
    agreeableness: Optional[int] = 50
    extraversion: Optional[int] = 50
    openness: Optional[int] = 70
    neuroticism: Optional[int] = 30
    steps: Optional[int] = None  # override global
    guidance: Optional[float] = None  # override global
    custom_prompt: Optional[str] = None  # skip engine, use raw prompt


app = FastAPI(title="ClawSouls Avatar API", version="1.0")


def check_auth(authorization: str = Header(None)):
    if authorization != f"Bearer {SECRET_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized: invalid or missing token")


# ── Model loaded at startup ─────────────────────────
@app.on_event("startup")
async def startup():
    global pipe
    variant = cfg["variant"]
    pipe = AutoPipelineForText2Image.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if variant == "fp16" else torch.float32,
        variant=variant if variant != "fp32" else None,
        use_safetensors=True,
    )
    pipe.enable_attention_slicing()
    if hasattr(pipe, 'enable_vae_tiling'):
        pipe.enable_vae_tiling()
    pipe.to("cuda")
    print(f\"✅ Modelo carregado: {MODEL_ID}\")


@app.get("/health")
def health():
    return {"status": "ok", "model": MODELO, "model_id": MODEL_ID}


@app.get("/models")
def list_models():
    return {"available": CATALOGO, "current": MODELO}


@app.post("/generate")
def generate(req: GenerateRequest, auth: str = Header(None)):
    check_auth(auth)

    # Build prompt
    if req.custom_prompt:
        prompt = req.custom_prompt
    else:
        prompt = build_prompt(req.dict())

    negative = build_negative_prompt()
    steps = req.steps if req.steps else TOTAL_STEPS
    guidance = req.guidance if req.guidance else TOTAL_GUIDANCE

    # Generate
    start = time.time()
    generator = torch.Generator(device="cuda").manual_seed(int(time.time() * 1000) % 2**32)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=guidance,
        width=cfg["width"],
        height=cfg["height"],
        generator=generator,
    ).images[0]
    elapsed = time.time() - start

    # Encode to base64
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    buf.seek(0)
    img_base64 = base64.b64encode(buf.read()).decode("utf-8")

    safe_name = "".join(c if c.isalnum() or c in "._-" else "_" for c in req.name.lower().strip())

    print(f"✅ [{req.name}] generated in {elapsed:.1f}s")

    return {
        "name": req.name,
        "slug": safe_name,
        "prompt": prompt,
        "negative_prompt": negative,
        "seed": generator.initial_seed(),
        "steps": steps,
        "guidance": guidance,
        "model": MODELO,
        "elapsed_s": round(elapsed, 1),
        "image_base64": img_base64,
    }


# ═══════════════════════════════════════════════════════
# START SERVER
# ═══════════════════════════════════════════════════════

print("=" * 60)
print("🚀 Iniciando servidor FastAPI...")
print("=" * 60)
%run -m uvicorn main:app --host 0.0.0.0 --port 8000 &

import time as _t
_t.sleep(3)  # wait for server to start

# ═══════════════════════════════════════════════════════
# CLOUDFLARED TUNNEL
# ═══════════════════════════════════════════════════════

!nohup cloudflared tunnel --url http://localhost:8000 > /tmp/cloudflared.log 2>&1 &

import time, requests, json, re

# Wait for tunnel URL
tunnel_url = None
for i in range(30):
    try:
        with open('/tmp/cloudflared.log', 'r') as f:
            log = f.read()
        match = re.search(r'https://[\w.-]+\.trycloudflare\.com', log)
        if match:
            tunnel_url = match.group(0)
            break
    except:
        pass
    time.sleep(1)

if tunnel_url:
    print('\n' + '=' * 60)
    print('✅ TÚNEL CLOUDFLARED ATIVO!')
    print(f'📎 URL pública: {tunnel_url}')
    print('\nEndpoints disponíveis:')
    print(f'  GET  {tunnel_url}/health')
    print(f'  GET  {tunnel_url}/models')
    print(f'  POST {tunnel_url}/generate')
    print('=' * 60)
else:
    print('⚠️  Não foi possível obter a URL do tunnel.')
    print('   Verifique: cat /tmp/cloudflared.log')

In [ ]:
# ═══════════════════════════════════════════════════════
# CÉLULA 7 — Testar a API
# ═══════════════════════════════════════════════════════

# Teste rápido — gera um avatar para o Jack

import requests, json, base64, IPython.display

if 'tunnel_url' in dir() and tunnel_url:
    url = tunnel_url + '/generate'

    payload = {
        "name": "Jack",
        "creature": "AI / Private Detective",
        "emoji": "🕵️",
        "vibe": "Detetive particular dos anos 40 adaptado para o digital. Perspicaz, irônico, vê através de mentiras.",
        "humor": 50,
        "formality": 50,
        "knowledgeDomains": [],
        "emotionalRange": 50,
        "vibeStyle": "concise"
    }

    headers = {"Authorization": f"Bearer {SECRET_TOKEN}", "Content-Type": "application/json"}

    r = requests.post(url, json=payload, headers=headers, timeout=120)

    if r.status_code == 200:
        data = r.json()
        print(f"✅ Geração: {data['name']} ({data['elapsed_s']}s)")
        print(f"   Prompt : {data['prompt'][:100]}...")
        print(f"   Seed   : {data['seed']}")

        # Decodificar e exibir a imagem
        img_data = base64.b64decode(data['image_base64'])
        IPython.display.display(IPython.display.Image(data=img_data))
    else:
        print(f"❌ Erro: {r.status_code} - {r.text}")
else:
    print("⚠️  URL do tunnel não disponível. Verifique a Célula 6.")

---

## Como usar a partir do Telegram / Agente

Quando o agente quiser gerar um avatar, ele faz:

```python
import requests, base64

# O agente recebe a URL do tunnel (você informa ou ele salva)
API_URL = "<sua-url-do-tunnel>.trycloudflare.com"

response = requests.post(
    f"{API_URL}/generate",
    json={
        "name": "Kira",
        "creature": "AI / Idol",
        "emoji": "🎤",
        "vibe": "Ídolo pop digital.",
        "humor": 50,
        "formality": 50,
        "knowledgeDomains": [],
        "emotionalRange": 60,
        "vibeStyle": "expressive"
    },
    headers={"Authorization": "Bearer clawsouls-secret"},
    timeout=120
)

data = response.json()
img = base64.b64decode(data["image_base64"])
# Salvar img e enviar via Telegram
```

### Parâmetros da requisição

| Campo | Tipo | Descrição |
|-------|------|-----------|
| `name` | string | Nome da soul (obrigatório) |
| `creature` | string | Tipo de criatura |
| `emoji` | string | Emoji visual |
| `vibe` | string | Descrição da personalidade |
| `humor` | int 0-100 | Nível de humor |
| `formality` | int 0-100 | Nível de formalidade |
| `knowledgeDomains` | list | Domínios de conhecimento |
| `emotionalRange` | int | Faixa emocional |
| `vibeStyle` | string | concise, minimal, expressive, etc. |
| `steps` | int (opt) | Override de steps |
| `guidance` | float (opt) | Override de guidance |
| `custom_prompt` | string (opt) | Prompt bruto (ignora engine) |

### Segurança
- Header `Authorization: Bearer <SECRET_TOKEN>` obrigatório
- Token configurado na Célula 5 (variável `SECRET_TOKEN`)
